# Any6D — Demo Verification & Visualization

Validates the full Any6D demo pipeline and produces visualizations entirely in Python.

**Architecture:**
```
master_env venv                    Docker any6d (--entrypoint "")
────────────────────────────       ──────────────────────────────
This notebook (Python viz)    -->  bash -c "python run_demo.py"
reads pose + projects mesh    <--  saves pose to results/
draws 3D bbox + pose axes
```

**Prerequisites:**
- `source ~/open-vocabulary-6d-pose-yoloe/master_env/bin/activate`
- Docker running with the `any6d` container
- Kernel = `master_env`

## Cell 1 — Config & Imports

In [ ]:
import os
import subprocess
import numpy as np
import cv2
import trimesh
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import sys

BASE_DIR  = os.path.expanduser('~/open-vocabulary-6d-pose-yoloe')
ANY6D_DIR = os.path.join(BASE_DIR, 'Any6D')
DEMO_DATA = os.path.join(ANY6D_DIR, 'demo_data')
RESULTS   = os.path.join(ANY6D_DIR, 'results', 'demo_mustard')
OBJ_ID    = 5  # mustard bottle in YCB

sys.path.insert(0, os.path.join(BASE_DIR, 'utils'))
from any6d_utils import (
    docker_run, check_docker,
    project_points, mesh_bbox_corners,
    draw_3d_bbox, draw_pose_axes,
    mask_overlay, colormap_depth,
    translation_error, rotation_error,
    axes_legend, save_fig
)

for name, path in [('ANY6D_DIR', ANY6D_DIR), ('DEMO_DATA', DEMO_DATA)]:
    assert os.path.exists(path), f'{name} not found: {path}'
    print(f'  OK  {name}: {path}')
os.makedirs(RESULTS, exist_ok=True)

## Cell 2 — Docker Check

In [ ]:
check_docker(ANY6D_DIR)

## Cell 3 — Run Demo (Mustard Bottle)

In [ ]:
print('Running run_demo.py in Docker...')

r = docker_run(
    ANY6D_DIR,
    'cd /workspace && python run_demo.py --ycb_model_path /workspace/demo_data',
    timeout=300
)

for line in r.stdout.split('\n'):
    if any(k in line.lower() for k in ['scale', 'chamfer', 'seed', 'candidates']):
        print(f'  {line.strip()}')

if r.returncode == 0:
    print('Demo completed')
else:
    print('Error:')
    print(r.stderr[-1000:])

## Cell 4 — Scene Image + Segmentation Mask

In [ ]:
color = cv2.cvtColor(cv2.imread(os.path.join(DEMO_DATA, 'color.png')), cv2.COLOR_BGR2RGB)
depth = cv2.imread(os.path.join(DEMO_DATA, 'depth.png'), cv2.IMREAD_ANYDEPTH).astype(np.float32) / 1000.0
label = np.load(os.path.join(DEMO_DATA, 'labels.npz'))

mask_bool   = (label['seg'] == OBJ_ID)
overlay     = mask_overlay(color, mask_bool)
depth_color = colormap_depth(depth)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(color);                          axes[0].set_title('Original Scene')
axes[1].imshow(mask_bool.astype(np.uint8)*255, cmap='gray'); axes[1].set_title(f'Segmentation Mask (object #{OBJ_ID})')
axes[2].imshow(overlay);                        axes[2].set_title('Scene + Mask Overlay')
axes[3].imshow(depth_color);                    axes[3].set_title(f'Depth Map [{depth.min():.2f}—{depth.max():.2f}] m')
for ax in axes: ax.axis('off')

plt.suptitle('Input Data — Mustard Bottle Demo')
plt.tight_layout()
save_fig(fig, os.path.join(RESULTS, 'input_visualization.png'))
plt.show()
print(f'Image : {color.shape}   Mask pixels: {mask_bool.sum()}')
print(f'Depth : {depth.shape}   range [{depth.min():.2f}, {depth.max():.2f}] m')

## Cell 5 — Pose Comparison & Bar Chart

In [ ]:
pred_pose = np.loadtxt(os.path.join(RESULTS, 'demo_mustard_initial_pose.txt'))
gt_pose   = np.loadtxt(os.path.join(RESULTS, 'demo_mustard_gt_pose.txt'))
K         = np.loadtxt(os.path.join(RESULTS, 'K.txt'))

t_pred = pred_pose[:3, 3]
t_gt   = gt_pose[:3, 3]
err_t  = translation_error(pred_pose, gt_pose)
err_R  = rotation_error(pred_pose, gt_pose)

print('=== POSE COMPARISON ===')
print(f'\n  Position (cm):      {"X":>8}   {"Y":>8}   {"Z":>8}')
print(f'  Predicted     :  {t_pred[0]*100:+8.1f}  {t_pred[1]*100:+8.1f}  {t_pred[2]*100:+8.1f}')
print(f'  Ground Truth  :  {t_gt[0]*100:+8.1f}  {t_gt[1]*100:+8.1f}  {t_gt[2]*100:+8.1f}')
print(f'\n  Translation error : {err_t:.2f} cm')
print(f'  Rotation error    : {err_R:.2f} deg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(3)
w = 0.35
b1 = axes[0].bar(x - w/2, t_pred*100, w, label='Predicted',    color='#2196F3', alpha=0.85)
b2 = axes[0].bar(x + w/2, t_gt*100,   w, label='Ground Truth', color='#4CAF50', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(['X (cm)', 'Y (cm)', 'Z (cm)'], fontsize=11)
axes[0].set_ylabel('cm', fontsize=11)
axes[0].set_title('Translation per Axis')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
for bar in list(b1) + list(b2):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

err_values = [err_t, err_R]
err_colors = ['#FF5722' if err_t > 5 else '#4CAF50', '#FF5722' if err_R > 10 else '#4CAF50']
bars = axes[1].bar(['Translation\nError (cm)', 'Rotation\nError (deg)'],
                   err_values, color=err_colors, alpha=0.85, width=0.4)
for bar, val in zip(bars, err_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=12)
axes[1].set_title('Estimation Errors (green = good)')
axes[1].set_ylabel('Error value', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Any6D — Pose Estimation Results')
plt.tight_layout()
save_fig(fig, os.path.join(RESULTS, 'pose_comparison.png'))
plt.show()

## Cell 6 — 6D Pose Visualized on Image

Project the 3D mesh bounding box and coordinate axes onto the scene. Pure NumPy + OpenCV + matplotlib.

In [ ]:
mesh       = trimesh.load(os.path.join(RESULTS, 'final_mesh_demo_mustard.obj'))
corners_3d = mesh_bbox_corners(mesh)

c_pred = project_points(corners_3d, pred_pose, K)
c_gt   = project_points(corners_3d, gt_pose,   K)
base   = overlay.copy()

img_pred = draw_pose_axes(draw_3d_bbox(base, c_pred, (30,120,255)), pred_pose, K)
img_gt   = draw_pose_axes(draw_3d_bbox(base, c_gt,   (30,200,80)),  gt_pose,   K)
img_both = draw_pose_axes(
    draw_3d_bbox(draw_3d_bbox(base, c_gt, (30,200,80)), c_pred, (30,120,255)),
    pred_pose, K
)

fig = plt.figure(figsize=(21, 7))
gs  = GridSpec(1, 3, figure=fig, wspace=0.04)

ax1 = fig.add_subplot(gs[0])
ax1.imshow(img_pred)
ax1.set_title(f'Predicted Pose — T err={err_t:.1f} cm  R err={err_R:.1f} deg')
ax1.axis('off')
axes_legend(ax1, [('#1E78FF','Predicted 3D bbox'),('#DC3232','X'),('#32D232','Y'),('#3264E6','Z')])

ax2 = fig.add_subplot(gs[1])
ax2.imshow(img_gt)
ax2.set_title('Ground Truth Pose')
ax2.axis('off')
axes_legend(ax2, [('#1EC850','Ground truth 3D bbox')])

ax3 = fig.add_subplot(gs[2])
ax3.imshow(img_both)
ax3.set_title('Overlay — Blue = Predicted  |  Green = GT')
ax3.axis('off')
axes_legend(ax3, [('#1E78FF','Predicted'),('#1EC850','Ground Truth')])

plt.suptitle('Any6D — 6D Pose on Mustard Bottle  |  Axes: Red=X  Green=Y  Blue=Z', y=1.01)
save_fig(fig, os.path.join(RESULTS, 'pose_visualization.png'))
plt.show()

## Cell 7 — Final Summary

In [ ]:
print('=' * 55)
print('ANY6D DEMO — FINAL SUMMARY')
print('=' * 55)
print(f'\n  Translation error : {err_t:.2f} cm')
print(f'  Rotation error    : {err_R:.2f} deg')
print('\n  Output files:')
for f in ['input_visualization.png', 'pose_comparison.png',
          'pose_visualization.png', 'demo_mustard_initial_pose.txt']:
    path   = os.path.join(RESULTS, f)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {f}')